# Análisis y Predicción de la Calidad del Aire en el GAM

**Institución:** Colegio Universitario de Cartago  
**Curso:** Big Data (BD-143) — III Cuatrimestre 2025  
**Profesor:** Osvaldo González Chaves  
**Estudiantes:** Claret Rodríguez | Mariana Méndez  

---

## Descripción General

Este notebook presenta el análisis exploratorio de datos (EDA), las visualizaciones y los resultados del modelo de Machine Learning desarrollado para predecir la categoría de calidad del aire en el Gran Área Metropolitana (GAM) de Costa Rica.

La pregunta central del proyecto es:

> **¿Cuando hay más vehículos en circulación, la calidad del aire en el GAM empeora?**

Para responder esta pregunta, se integran tres fuentes principales de datos:

- **Flujo vehicular** de la Ruta 27 (ARESEP) — 2009 a 2025  
- **Calidad del aire** (PM2.5, PM10, NO2, CO, Ozono) — Open-Meteo API  
- **Datos climáticos históricos** (temperatura, humedad, viento) — Open-Meteo Archive API  

Estas fuentes permiten analizar la relación entre movilidad urbana, condiciones ambientales y niveles de contaminación.

---

## Variable Objetivo

El modelo busca predecir la **categoría del ICA (Índice Costarricense de Calidad del Aire)**, definida a partir de los niveles de PM2.5:

| Categoría | PM2.5 (μg/m³) | Descripción |
|-----------|--------------|-------------|
| Buena | 0 — 12 | Calidad del aire satisfactoria |
| Moderada | 12 — 35 | Calidad aceptable |
| Mala | 35 — 55 | Grupos sensibles pueden verse afectados |
| Muy Mala | +55 | Toda la población puede verse afectada |

In [1]:
# ─────────────────────────────────────────────────────────────
# Importaciones y configuración del entorno
# ─────────────────────────────────────────────────────────────

import sys
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de visualización
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Agregar src al path para importar módulos del proyecto
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))

# Importar clases del proyecto
from basedatos.gestor_base_datos import GestorBaseDatos
from eda.procesador_eda import ProcesadorEDA
from visualizacion.visualizador import Visualizador
from modelos.modelo_ml import ModeloML

print(" Librerías importadas correctamente")
print(f" Directorio de trabajo: {os.getcwd()}")

 Librerías importadas correctamente
 Directorio de trabajo: C:\Users\mari2\Desktop\Progra 2\Proyecto Final\notebooks


## 1. Carga de Datos desde SQL Server

Los datos utilizados en este proyecto fueron previamente procesados y almacenados en una base de datos **SQL Server** llamada `CalidadAireGAM`.

Se dispone de tres tablas principales:

| Tabla | Descripción | Registros |
|-------|-------------|-----------|
| `flujo_vehicular` | Flujo vehicular mensual en la Ruta 27 (ARESEP), periodo 2014–2024 | 1,440 |
| `calidad_aire` | Variables de calidad del aire (PM2.5, PM10, NO2, CO, Ozono) obtenidas mediante la API de Open-Meteo (2022–2024) | 29 |
| `clima` | Variables climáticas (temperatura, humedad, velocidad del viento) (2022–2024) | 29 |

Estas fuentes de datos permiten integrar información de movilidad, contaminación y condiciones climáticas para su posterior análisis.

A continuación, se realiza la carga de los datos directamente desde la base de datos utilizando la clase `GestorBaseDatos`.

In [2]:
# ─────────────────────────────────────────────────────────────
# CELDA 4: Carga de datos desde SQL Server
# ─────────────────────────────────────────────────────────────

# Inicializar conexión a la base de datos
gestor_bd = GestorBaseDatos()

# Cargar las tres tablas
df_flujo = gestor_bd.consultar("SELECT * FROM flujo_vehicular")
df_aire = gestor_bd.consultar("SELECT * FROM calidad_aire")
df_clima = gestor_bd.consultar("SELECT * FROM clima")

# Mostrar resumen de cada tabla
print("=" * 55)
print(" DATOS CARGADOS DESDE SQL SERVER")
print("=" * 55)
print(f"\n Flujo Vehicular: {df_flujo.shape[0]} filas x {df_flujo.shape[1]} columnas")
print(f"  Calidad del Aire: {df_aire.shape[0]} filas x {df_aire.shape[1]} columnas")
print(f"  Clima: {df_clima.shape[0]} filas x {df_clima.shape[1]} columnas")

print("\n--- Vista previa: Flujo Vehicular ---")
display(df_flujo.head())

print("\n--- Vista previa: Calidad del Aire ---")
display(df_aire.head())

print("\n--- Vista previa: Clima ---")
display(df_clima.head())

[GestorBaseDatos] Error al conectar: (pyodbc.OperationalError) ('08001', '[08001] [Microsoft][ODBC Driver 17 for SQL Server]SQL Server Network Interfaces: Error Locating Server/Instance Specified [xFFFFFFFF].  (-1) (SQLDriverConnect); [08001] [Microsoft][ODBC Driver 17 for SQL Server]Login timeout expired (0); [08001] [Microsoft][ODBC Driver 17 for SQL Server]A network-related or instance-specific error has occurred while establishing a connection to SQL Server. Server is not found or not accessible. Check if instance name is correct and if SQL Server is configured to allow remote connections. For more information see SQL Server Books Online. (-1)')
(Background on this error at: https://sqlalche.me/e/20/e3q8)
[GestorBaseDatos] Error en consulta: Query must be a string unless using sqlalchemy.
[GestorBaseDatos] Error en consulta: Query must be a string unless using sqlalchemy.
[GestorBaseDatos] Error en consulta: Query must be a string unless using sqlalchemy.
 DATOS CARGADOS DESDE SQL 

AttributeError: 'NoneType' object has no attribute 'shape'

## 2. Análisis Exploratorio de Datos (EDA)

En esta sección se realiza el análisis exploratorio de los datos con el objetivo de comprender su estructura, calidad y comportamiento general.

Se analizan tres conjuntos de datos principales:

- **Flujo vehicular**  
- **Calidad del aire**  
- **Variables climáticas**  

Los cuales fueron previamente almacenados en una base de datos SQL Server.

A través de la clase `ProcesadorEDA`, se llevan a cabo los siguientes procesos:

### Procesos realizados

- Análisis general de cada dataset, incluyendo dimensiones, tipos de datos y estadísticas descriptivas  
- Identificación de valores nulos y posibles inconsistencias  
- Análisis temporal del comportamiento de las variables  
- Evaluación de la relación entre el flujo vehicular y los niveles de contaminación  

### Objetivo del análisis

Este análisis permite obtener una visión inicial de los datos, identificar patrones relevantes y detectar posibles problemas en la calidad de la información. Además, sirve como base para las etapas posteriores de visualización y modelado predictivo.

In [ ]:
# Crear instancia del procesador EDA
eda = ProcesadorEDA()

# Asignar los DataFrames ya cargados desde SQL Server
eda.df_flujo = df_flujo
eda.df_aire = df_aire
eda.df_clima = df_clima

# =========================================
# RESUMEN GENERAL DE LOS DATOS
# =========================================
eda.resumen_general(eda.df_flujo, "Flujo Vehicular")
eda.resumen_general(eda.df_aire, "Calidad del Aire")
eda.resumen_general(eda.df_clima, "Clima")

# =========================================
# ANÁLISIS DE VALORES NULOS
# =========================================
eda.valores_nulos(eda.df_flujo, "Flujo Vehicular")
eda.valores_nulos(eda.df_aire, "Calidad del Aire")
eda.valores_nulos(eda.df_clima, "Clima")

# =========================================
# ANÁLISIS TEMPORAL
# =========================================
df_flujo_anio = eda.flujo_por_anio()
df_aire_anio = eda.promedio_aire_por_anio()
df_clima_anio = eda.promedio_clima_por_anio()

# =========================================
# CORRELACIÓN ENTRE VARIABLES
# =========================================
correlacion = eda.correlacion_flujo_aire()

### Resultados del Análisis Exploratorio

A partir del análisis realizado, se pueden destacar los siguientes hallazgos:

#### Estructura de los datos
- El dataset de flujo vehicular es el más grande, con 1440 registros, mientras que los datasets de calidad del aire y clima contienen 29 registros cada uno.
- Esto indica que los datos de aire y clima están agregados a nivel mensual, mientras que el flujo vehicular tiene mayor detalle.

#### Calidad de los datos
- No se encontraron valores nulos en ninguno de los datasets, lo cual facilita el análisis y modelado posterior.
- Las variables presentan tipos de datos adecuados (enteros y flotantes), por lo que no se requieren transformaciones adicionales en esta etapa.

#### Análisis temporal
- El flujo vehicular muestra variaciones a lo largo de los años, con un aumento general hasta 2019 y una disminución en años posteriores.
- Los contaminantes como PM2.5 y NO2 presentan variaciones moderadas entre los años 2022 y 2024.
- Las variables climáticas se mantienen relativamente estables, con ligeras variaciones en temperatura, humedad y velocidad del viento.

#### Correlación entre variables
- Se observa una correlación positiva entre el flujo vehicular y el PM2.5 (0.30), lo que sugiere que a mayor cantidad de vehículos, mayor concentración de partículas contaminantes.
- También existe una relación positiva entre PM2.5 y dióxido de nitrógeno (NO2).
- El ozono presenta una correlación negativa con el NO2, lo cual es consistente con procesos atmosféricos conocidos.

#### Conclusión del EDA
En general, los resultados sugieren que el flujo vehicular tiene una influencia moderada en la calidad del aire, aunque no es el único factor determinante.  
Las condiciones climáticas también juegan un papel importante en la dispersión y concentración de contaminantes.

Este análisis servirá como base para la construcción del modelo de Machine Learning en la siguiente sección.

## 3. Visualización de Datos

En esta sección se presentan diferentes visualizaciones con el objetivo de analizar el comportamiento del flujo vehicular, la calidad del aire y las variables climáticas.

Las gráficas permiten identificar patrones, tendencias y relaciones entre las variables, facilitando la interpretación de los datos obtenidos en el análisis exploratorio.

### Visualizaciones realizadas

- Flujo vehicular total por año
- Flujo vehicular promedio por mes
- Niveles de PM2.5 por mes
- Comparación de contaminantes por año
- Mapa de correlación entre variables
- Tendencia de temperatura
- Mapa geográfico de la Ruta 27

Estas visualizaciones ayudan a comprender mejor la relación entre el tráfico vehicular y la calidad del aire en el GAM.

In [ ]:
viz = Visualizador()

# Asignar los datos ya cargados
viz.df_flujo = df_flujo
viz.df_aire = df_aire
viz.df_clima = df_clima

# =========================================
# GRÁFICOS
# =========================================

In [ ]:
viz.grafico_flujo_por_anio()

### Interpretación

Se observa que el flujo vehicular se mantiene relativamente estable a lo largo de los años, con ligeras variaciones entre periodos. Sin embargo, se identifica una caída notable en el año 2021, la cual podría estar relacionada con factores externos como restricciones de movilidad asociadas a la pandemia. Posteriormente, el flujo muestra una recuperación en los años siguientes, aunque sin alcanzar completamente los niveles máximos observados en años anteriores como 2019.

### Insight

El comportamiento del flujo vehicular no sigue una tendencia lineal creciente, lo que indica que está fuertemente influenciado por factores externos como eventos sociales, económicos o sanitarios. Además, la caída en 2021 y su posterior recuperación sugieren que la movilidad es sensible a cambios en el entorno, lo cual es relevante para analizar su impacto en la calidad del aire y considerar estos eventos en modelos predictivos.

In [ ]:
viz.grafico_flujo_por_mes()

### Interpretación

El flujo vehicular promedio por mes presenta variaciones a lo largo del año, lo que evidencia la existencia de patrones estacionales. Se observan meses con mayor circulación, como marzo, octubre y diciembre, mientras que otros como febrero y junio muestran niveles más bajos. Estas diferencias podrían estar asociadas a factores como actividades laborales, periodos vacacionales o condiciones climáticas que influyen en la movilidad.

### Insight

El flujo vehicular presenta un comportamiento estacional claro, lo que indica que la movilidad no es uniforme durante el año. Esto es relevante para el análisis de la calidad del aire, ya que en los meses con mayor circulación podrían registrarse mayores niveles de contaminación. Por lo tanto, es importante considerar la variable temporal (mes) en modelos predictivos, ya que puede mejorar la capacidad de explicar y anticipar cambios en la calidad del aire.

In [ ]:
viz.grafico_pm25_por_mes()

### Interpretación

Los niveles de PM2.5 presentan variaciones mensuales a lo largo de los años analizados, evidenciando un comportamiento no uniforme. Se observan picos más altos en meses como marzo, abril y mayo, especialmente en 2023, mientras que hacia finales del año (noviembre y diciembre) los niveles tienden a disminuir. Además, aunque los tres años siguen un patrón similar, existen diferencias en la magnitud de las concentraciones, lo que indica variabilidad interanual.

### Insight

La variación del PM2.5 a lo largo del año sugiere la influencia de factores estacionales y ambientales, como condiciones climáticas, lluvias o cambios en la dispersión de contaminantes. Aunque el flujo vehicular puede contribuir a estos niveles, las diferencias entre años indican que no es el único factor determinante. Esto resalta la importancia de incluir variables adicionales en el análisis, ya que la calidad del aire depende de múltiples factores y no únicamente del tráfico.

In [ ]:
viz.grafico_contaminantes_por_anio()

### Interpretación

Los contaminantes analizados (PM2.5, dióxido de nitrógeno y ozono) presentan comportamientos diferenciados a lo largo de los años. El PM2.5 se mantiene relativamente estable, mientras que el dióxido de nitrógeno muestra una tendencia decreciente desde 2022 hasta 2024. Por otro lado, el ozono presenta un aumento progresivo significativo, convirtiéndose en el contaminante con mayor crecimiento en el periodo analizado.

### Insight

El comportamiento diferenciado de los contaminantes sugiere que cada uno está influenciado por fuentes y procesos distintos. La disminución del dióxido de nitrógeno podría estar asociada a cambios en el flujo vehicular o mejoras en emisiones, mientras que el aumento del ozono indica la posible influencia de factores atmosféricos y reacciones químicas en el ambiente. Esto refuerza que la calidad del aire es un fenómeno complejo y multifactorial, por lo que no puede explicarse únicamente a partir de una sola variable como el tráfico vehicular.

In [ ]:
viz.heatmap_correlacion()

### Interpretación

La matriz de correlación muestra que la relación entre el flujo vehicular y los contaminantes es en general baja a moderada. El PM2.5 presenta una correlación positiva débil con el flujo vehicular (0.31), al igual que el dióxido de nitrógeno (0.22), lo que sugiere cierta asociación entre el tráfico y estos contaminantes.

Por otro lado, el ozono presenta una correlación negativa con el flujo vehicular (-0.23), lo que indica un comportamiento diferente en comparación con los otros contaminantes. Además, se observa una correlación negativa fuerte entre el dióxido de nitrógeno y el ozono (-0.85), lo que evidencia una relación inversa significativa entre ambos.

### Insight

Aunque el flujo vehicular influye en ciertos contaminantes como el PM2.5 y el dióxido de nitrógeno, las correlaciones relativamente bajas indican que no es el único factor determinante en la calidad del aire. La fuerte relación inversa entre el dióxido de nitrógeno y el ozono sugiere la presencia de procesos químicos en la atmósfera que afectan la formación de contaminantes secundarios.

En particular, el ozono es un contaminante secundario que se forma a partir de reacciones químicas en el ambiente, lo que explica su comportamiento distinto frente al tráfico vehicular.

Esto evidencia que la calidad del aire es un fenómeno complejo y multifactorial, por lo que es necesario considerar variables adicionales, como condiciones climáticas y factores ambientales, para lograr modelos predictivos más precisos.

In [ ]:
viz.grafico_temperatura_por_anio()

### Interpretación

La temperatura promedio muestra una ligera tendencia al alza a lo largo de los años analizados. Se observa un incremento desde 2022 hasta 2023, seguido de una estabilización con un leve aumento en 2024. Aunque las variaciones no son muy pronunciadas, el comportamiento sugiere un incremento progresivo en la temperatura promedio en el periodo estudiado.

### Insight

Aunque la temperatura presenta cierta estabilidad relativa, la tendencia creciente podría influir indirectamente en la calidad del aire, ya que condiciones más cálidas pueden favorecer la formación de contaminantes como el ozono. Sin embargo, dado que los cambios son leves, es probable que otras variables climáticas como el viento, la humedad o la precipitación tengan un impacto más significativo en la dispersión o concentración de contaminantes.

In [ ]:
# =========================================
# MAPA
# =========================================

mapa = viz.mapa_ruta_27()
mapa

### Interpretación

El mapa permite ubicar geográficamente la Ruta 27 y los puntos de análisis dentro del Gran Área Metropolitana (GAM) de Costa Rica. Se observa que los puntos se distribuyen a lo largo de un corredor vial importante que conecta diferentes zonas, lo que evidencia su relevancia en el flujo vehicular y en la generación de datos para el análisis.

### Insight

La visualización geográfica permite comprender mejor el contexto espacial del estudio, destacando que la Ruta 27 es un eje clave de movilidad en el país. Esto es relevante, ya que la ubicación de los puntos de medición puede influir en los niveles de contaminación observados, considerando factores como densidad vehicular, cercanía a zonas urbanas y condiciones del entorno.

## Interpretación de las visualizaciones

A partir de las gráficas generadas, se pueden observar los siguientes aspectos:

- El flujo vehicular presenta variaciones a lo largo de los años, con cambios asociados a factores externos que afectan la movilidad, como eventos sociales o económicos.

- A nivel mensual, se identifican patrones estacionales en la circulación de vehículos, evidenciando que la movilidad no es uniforme durante el año.

- Los niveles de PM2.5 muestran fluctuaciones moderadas, con variabilidad entre meses y años, lo que refleja cambios en la calidad del aire.

- El heatmap de correlación evidencia una relación positiva débil entre el flujo vehicular y algunos contaminantes, así como relaciones inversas relevantes entre variables como el dióxido de nitrógeno y el ozono.

- La temperatura presenta una ligera tendencia al alza, lo que puede influir indirectamente en la formación y dispersión de contaminantes.

- El mapa permite ubicar geográficamente la Ruta 27 y los puntos de análisis, aportando contexto espacial al estudio.

En conjunto, estas visualizaciones muestran que el flujo vehicular tiene una influencia en la calidad del aire, pero no es el único factor determinante. La contaminación atmosférica es un fenómeno complejo y multifactorial, influenciado también por condiciones climáticas, procesos químicos y características del entorno.

## 4. Modelo de Machine Learning

En esta sección se desarrolla un modelo de clasificación con el objetivo de predecir la categoría del Índice de Calidad del Aire (ICA) a partir de variables relacionadas con el flujo vehicular, la calidad del aire y las condiciones climáticas.

Se utilizan técnicas de aprendizaje supervisado, en las cuales el modelo aprende patrones a partir de datos históricos para posteriormente realizar predicciones sobre nuevas observaciones.

### Objetivo del modelo

Predecir la categoría del Índice de Calidad del Aire (ICA) utilizando variables explicativas como:

- Flujo vehicular total  
- Dióxido de nitrógeno (NO2)  
- Ozono  
- Temperatura  
- Humedad relativa  
- Velocidad del viento  

> Nota: La variable PM2.5 no se utiliza como variable de entrada, ya que a partir de ella se construye la variable objetivo (categoría ICA).

### Enfoque

Se entrenan y evalúan distintos modelos de Machine Learning con el fin de comparar su desempeño en la tarea de clasificación. Esto permite identificar cuál modelo ofrece mejores resultados y mayor capacidad predictiva según las características del conjunto de datos.

Debido al desbalance en las clases y la baja cantidad de datos en algunas categorías, los resultados se interpretan como una aproximación inicial del comportamiento del fenómeno.

In [ ]:
# =========================================
# INICIALIZAR MODELO
# =========================================
modelo = ModeloML()

# =========================================
# PREPARACIÓN DE DATOS
# =========================================
df_modelo = modelo.cargar_y_preparar_datos()

# División en entrenamiento y prueba
modelo.dividir_datos()

# =========================================
# ENTRENAMIENTO DE MODELOS
# =========================================
modelo.entrenar_random_forest()
modelo.entrenar_arbol_decision()
modelo.entrenar_knn()
modelo.entrenar_regresion_logistica()

# =========================================
# COMPARACIÓN DE MODELOS
# =========================================
modelo.comparar_modelos()

# =========================================
# NOTA SOBRE VALIDACIÓN
# =========================================
print("\n[INFO] Cross-validation y GridSearch no se aplican debido al desbalance de clases.\n")

# =========================================
# EJEMPLO DE PREDICCIÓN
# =========================================
modelo.predecir_nuevo(
    total_vehiculos=950000,
    nitrogen_dioxide=15.2,
    ozone=32.1,
    temperature_2m=20.5,
    relative_humidity_2m=78.0,
    windspeed_10m=7.5
)

## Resultados del modelo

Los modelos entrenados presentan un accuracy del 100%, lo que inicialmente podría interpretarse como un excelente desempeño. Sin embargo, al analizar con mayor detalle los resultados, se observa que el dataset está altamente desbalanceado.

La mayoría de los registros pertenecen a la categoría **"Buena"** (28 observaciones), mientras que únicamente existe una observación en la categoría **"Moderada"**. Esta distribución desigual provoca que los modelos tiendan a predecir principalmente la clase mayoritaria.

Además, el conjunto de prueba contiene únicamente registros de la categoría **"Buena"**, lo que facilita que el modelo obtenga una predicción perfecta sin necesariamente haber aprendido patrones complejos.

---

## Análisis

A partir de los resultados obtenidos, se identifican los siguientes aspectos:

- El modelo clasifica correctamente la mayoría de los casos correspondientes a la categoría **"Buena"**.
- No existe capacidad real de evaluación sobre la categoría **"Moderada"**, debido a la falta de representación en el conjunto de prueba.
- El accuracy no es una métrica suficiente en este contexto, ya que está influenciado por el desbalance de clases.
- No fue posible aplicar validación cruzada (Cross-Validation) ni optimización de hiperparámetros (GridSearchCV), debido a la baja cantidad de datos en la clase minoritaria.

---

## Corrección del modelo

Inicialmente se utilizó la variable PM2.5 como variable de entrada del modelo.

Sin embargo, se identificó que esto generaba un problema de fuga de información, ya que la variable objetivo (categoría ICA) se construye directamente a partir de los valores de PM2.5.

Esto provocaba que el modelo tuviera acceso indirecto a la respuesta correcta, generando resultados artificialmente altos.

Por esta razón, se decidió eliminar PM2.5 de las variables de entrada, manteniendo únicamente variables independientes como:

- Flujo vehicular  
- NO2  
- Ozono  
- Temperatura  
- Humedad  
- Velocidad del viento  

Con esta corrección, el modelo se vuelve más realista y alineado con buenas prácticas de Machine Learning.

---

## Conclusión

El modelo evidencia que existe cierta relación entre las variables analizadas (flujo vehicular y condiciones climáticas) y la calidad del aire.

No obstante, debido al tamaño reducido del dataset y al fuerte desbalance entre las categorías del ICA, los resultados deben interpretarse con precaución. El modelo presenta limitaciones importantes para generalizar y detectar correctamente clases minoritarias.

Para mejorar el desempeño del modelo, sería necesario:

- Contar con un mayor volumen de datos  
- Lograr una distribución más equilibrada entre las categorías del ICA  
- Considerar técnicas de balanceo de datos o ajustes en las métricas de evaluación  

En su estado actual, el modelo permite identificar patrones generales, pero no es suficiente para realizar predicciones completamente confiables en un entorno real.

## 5. Conclusiones

A partir del análisis realizado, se puede concluir que existe una relación entre el flujo vehicular y la calidad del aire en el Gran Área Metropolitana (GAM), aunque este no es el único factor que influye en el comportamiento de los contaminantes.

El análisis exploratorio y las visualizaciones permitieron identificar patrones relevantes en los datos, como variaciones temporales, comportamientos estacionales y relaciones entre diferentes contaminantes. Asimismo, el modelo de Machine Learning evidenció que variables como el flujo vehicular, los niveles de NO2 y ozono, así como las condiciones climáticas, tienen influencia en la calidad del aire.

No obstante, los resultados del modelo deben interpretarse con precaución debido a ciertas limitaciones:

- El tamaño reducido del dataset  
- El desbalance en la variable objetivo, con predominio de la categoría **"Buena"**  
- La imposibilidad de aplicar técnicas como Cross-Validation y GridSearchCV debido a la baja cantidad de datos en la clase minoritaria  
- La falta de variables adicionales que permitan capturar mejor la complejidad del fenómeno  

Estas limitaciones afectan la capacidad del modelo para generalizar y predecir correctamente otras categorías de calidad del aire, especialmente aquellas con menor representación.

Además, durante el desarrollo del modelo se identificó la importancia de evitar la fuga de información. Inicialmente se utilizó la variable PM2.5 como entrada, pero esta fue eliminada al ser utilizada para construir la variable objetivo, permitiendo así obtener un modelo más realista y alineado con buenas prácticas de Machine Learning.

En términos generales, este proyecto demuestra cómo el uso de herramientas de análisis de datos y Machine Learning permite comprender mejor fenómenos ambientales complejos y su relación con la movilidad urbana.

### Trabajo futuro

Para mejorar el alcance y la calidad del análisis, se proponen las siguientes líneas de trabajo:

- Incorporar un mayor volumen de datos históricos  
- Aplicar técnicas de balanceo de datos para reducir el sesgo  
- Incluir nuevas variables como tipo de vehículo o información por hora  
- Explorar modelos más avanzados y optimización de hiperparámetros  
- Utilizar métricas adicionales al accuracy para una evaluación más completa  

En conjunto, estas mejoras permitirían desarrollar modelos más robustos, precisos y aplicables en escenarios reales.